# RSP ILP Solver — `mp.ipynb`

**Purpose:** Use the `rsp` SDK to solve the Reliable Shortest Path problem with ILP.

Covers:
- Load a road network dataset
- Configure solver parameters (inline, with optional YAML loading)
- Solve a single OD pair — inspect the path, punctuality, and timing
- Run across multiple OD pairs — compare results
- Display and visualize the optimal paths

**Prerequisite:** Run from the repo root (`/home/kkk/projects/RSP`).
Defaults to `data/small` (10 nodes, 20 edges, 50 samples) for fast iteration.

## 1. Setup — Imports & Path

In [53]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root — works from workspace root or notebook/ dir
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebook" and (REPO_ROOT.parent / "rsp" / "__init__.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rsp import RSPDataset, RSPConfig, RSPRunner

print(f"Repo root : {REPO_ROOT}")

Repo root : /home/kkk/projects/RSP


## 2. Configuration

Pick a dataset and construct an `RSPConfig` inline. (Commented: how to load from a YAML file instead.)

In [ ]:
# ============================================================
# Dataset — uncomment one
# ============================================================
DATASET_PATH = str(REPO_ROOT / "Cao_SOTA_MP/data/small")               # 10 nodes,  fast debug
# DATASET_PATH = str(REPO_ROOT / "Cao_SOTA_MP/data/full/seed42")       # 65 nodes,  full experiment
# DATASET_PATH = str(REPO_ROOT / "Cao_SOTA_MP/data/beijing")           # 587 nodes, Beijing OSM
# DATASET_PATH = str(REPO_ROOT / "Cao_SOTA_MP/data/beijing_conflict")  # 587 nodes, conflict variance

# ============================================================
# Config — inline construction
# ============================================================
config = RSPConfig(
    methods=("ILP",),          # only ILP
    alphas=(0.5, 0.7, 0.9),    # deadline levels: low → tight, high → loose
    num_repeats=1,             # how many travel-time repeats to use
    num_od_pairs=3,            # how many OD pairs to run (None = all)
    solver_backend="SCIP",     # SCIP / CBC / GLPK
    time_limit=60,             # per-job solver timeout (seconds)
    deadline_mode="heuristic", # heuristic / exact
)

# ============================================================
# Alternative: load config from a YAML file
# ============================================================
# config = RSPConfig.from_yaml(str(REPO_ROOT / "Cao_SOTA_MP/configs/beijing.yaml"))
# config = RSPConfig.from_yaml(str(REPO_ROOT / "configs/beijing_conflict_300.yaml"))

print(f"Dataset   : {DATASET_PATH}")
print(f"Methods   : {list(config.methods)}")
print(f"Alphas    : {list(config.alphas)}")
print(f"Solver    : {config.solver_backend}")
print(f"Deadline  : {config.deadline_mode}")
print(f"Jobs      : {config.expected_job_count}")

## 3. Load Dataset

`RSPDataset.from_directory()` loads the standard layout and validates everything automatically.

In [55]:
dataset = RSPDataset.from_directory(DATASET_PATH)

print(f"Nodes      : {dataset.network.num_nodes}")
print(f"Edges      : {dataset.network.num_edges}")
print(f"Repeats    : {dataset.num_repeats}")
print(f"OD pairs   : {dataset.num_od_pairs}")
print(f"Samples (N): {dataset.travel_times[0].shape[0]}")
print(f"OD list    : {dataset.od_pairs}")

FileNotFoundError: [Errno 2] No such file or directory: 'Cao_SOTA_MP/data/small/network.npz'

### 3a. Inspect Network & Travel Times

In [ ]:
W0 = dataset.travel_times[0]
edge_means = W0.mean(axis=0)
edge_cvs = W0.std(axis=0) / edge_means

print(f"Edge mean travel time: [{edge_means.min():.1f}, {edge_means.max():.1f}] min")
print(
    f"Edge CV  — median: {np.median(edge_cvs):.3f}, "
    f"range: [{edge_cvs.min():.3f}, {edge_cvs.max():.3f}]"
)
print(f"N (samples per repeat): {W0.shape[0]}")
print(f"E (edges): {W0.shape[1]}")

## 4. Single OD — Solve One Case

Pick one `(repeat, od_idx, alpha)` and run ILP. Inspect the result.

In [ ]:
runner = RSPRunner(dataset, config)

# Pick a specific OD and alpha
REPEAT = 0
OD_IDX = 0
ALPHA = 0.7

origin, dest = dataset.od_pairs[OD_IDX]
case = runner.solve_case(repeat=REPEAT, od_idx=OD_IDX, alpha=ALPHA)

N = dataset.travel_times[REPEAT].shape[0]
ilp = case.ilp

print(f"OD        : {origin} → {dest}")
print(f"α         : {ALPHA}")
print(f"τ (tau)   : {case.tau:.2f}")
print(f"Status    : {ilp['status']}")
print(f"Punctuality: {ilp['punctuality_prob']:.4f}  ({ilp['lateness_count']}/{N} late)")
print(f"Solve time : {ilp['solve_time']:.4f} s")

### 4a. Display the Optimal Path

In [ ]:
if ilp["path_x"] is not None:
    # Convert binary edge vector → edge list
    path_edges = [
        dataset.edge_order[j]
        for j in np.where(ilp["path_x"] > 0.5)[0]
    ]

    # Build node sequence from edges
    node_seq = [path_edges[0][0]]
    for u, v in path_edges:
        node_seq.append(v)

    print(f"ILP optimal path ({len(path_edges)} edges, {len(path_edges)+1} nodes):")
    print(f"  Nodes: {' → '.join(str(n) for n in node_seq)}")

    # Show each edge with index in the edge_order
    for j, (u, v) in enumerate(path_edges):
        edge_idx = dataset.edge_order.index((u, v))
        mean_t = dataset.travel_times[REPEAT][:, edge_idx].mean()
        print(f"  Edge {j}: ({u}→{v})  [edge_order idx={edge_idx}, mean_time={mean_t:.1f}]")

    # Path travel-time distribution
    W = dataset.travel_times[REPEAT]
    path_times = W @ ilp["path_x"]
    tau = case.tau
    print(f"\nPath travel time stats:")
    print(f"  Mean: {path_times.mean():.1f}   Median: {np.median(path_times):.1f}")
    print(f"  Min:  {path_times.min():.1f}   Max:    {path_times.max():.1f}")
    print(f"  Late: {(path_times > tau).sum()}/{N}  (>{tau:.1f})")
    print(f"  Punctuality: {ilp['punctuality_prob']:.4f}")
else:
    print(f"ILP did not return a path. Status: {ilp['status']}")

### 4b. Path Travel-Time Histogram

In [ ]:
if ilp["path_x"] is not None:
    W = dataset.travel_times[REPEAT]
    path_times = W @ ilp["path_x"]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(path_times, bins=30, color="#3b82f6", edgecolor="white", alpha=0.85)
    ax.axvline(
        case.tau,
        color="#ef4444",
        linestyle="--",
        linewidth=2,
        label=f"τ = {case.tau:.1f}",
    )
    ax.set_xlabel("Travel Time")
    ax.set_ylabel("Samples")
    ax.set_title(
        f"ILP Path Travel-Time Distribution\n"
        f"OD ({origin}→{dest}), α={ALPHA}, "
        f"punctuality={ilp['punctuality_prob']:.3f}"
    )
    ax.legend()
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    plt.show()

## 5. Multi-OD — Run Across Multiple Pairs

Run ILP across all configured `(repeat × OD × alpha)` combinations.

In [ ]:
import time

t0 = time.perf_counter()
result = runner.run(progress=True)
elapsed = time.perf_counter() - t0

print(f"\nDone in {elapsed:.1f}s — {len(result.df)} rows")

### 5a. Results — Summary by Alpha

In [ ]:
by_alpha = result.by_alpha()
print("=== ILP Results by Alpha ===")
display(by_alpha.round(4))

# Punctuality vs alpha
pivot = by_alpha.pivot(index="alpha", columns="method", values="mean_punctuality_prob")
print("\n=== Mean Punctuality vs Alpha ===")
display(pivot)

### 5b. Results — By OD Pair

In [ ]:
by_od = result.by_od()
print("=== ILP Results by OD Pair ===")
display(by_od.round(4))

### 5c. Solve Time & Status

In [ ]:
print("=== Solve Time ===")
display(result.solve_time().round(4))

print("\n=== Solver Status ===")
display(result.status_counts())

### 5d. Enumerate All Optimal Paths

For each OD pair and alpha, show the ILP optimal path and its punctuality.

In [ ]:
df = result.to_dataframe()
alpha_list = sorted(df["alpha"].unique())

for od_idx in sorted(df["od_idx"].unique()):
    od_rows = df[df["od_idx"] == od_idx]
    origin = int(od_rows["origin"].iloc[0])
    destination = int(od_rows["destination"].iloc[0])

    print(f"\n{'='*60}")
    print(f"OD {od_idx}: {origin} → {destination}")
    print(f"{'='*60}")

    for alpha in alpha_list:
        # Re-run solve_case to get the actual path_x for display
        case_m = runner.solve_case(repeat=0, od_idx=od_idx, alpha=alpha)
        ilp_r = case_m.ilp

        if ilp_r is None or ilp_r.get("path_x") is None:
            print(f"  α={alpha:.1f}  NO PATH  status={ilp_r.get('status') if ilp_r else 'N/A'}")
            continue

        # Decode path
        path_edges = [
            dataset.edge_order[j]
            for j in np.where(ilp_r["path_x"] > 0.5)[0]
        ]
        node_seq = [str(path_edges[0][0])]
        for u, v in path_edges:
            node_seq.append(str(v))

        prob = ilp_r["punctuality_prob"]
        tau_val = case_m.tau

        print(f"\n  α={alpha:.1f}  τ={tau_val:.1f}  "
              f"punct={prob:.3f}  ({ilp_r['lateness_count']}/{N} late)  "
              f"time={ilp_r['solve_time']:.3f}s")
        print(f"  Path ({len(path_edges)} edges): {' → '.join(node_seq)}")
        print(f"  Edges: {path_edges}")

### 5e. Punctuality vs Alpha Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for od_idx in sorted(df["od_idx"].unique()):
    od_rows = df[(df["od_idx"] == od_idx) & (df["method"] == "ILP")]
    origin = int(od_rows["origin"].iloc[0])
    dest = int(od_rows["destination"].iloc[0])
    od_rows = od_rows.sort_values("alpha")
    ax.plot(
        od_rows["alpha"],
        od_rows["punctuality_prob"],
        "o-",
        label=f"OD{od_idx} ({origin}→{dest})",
        markersize=6,
    )

ax.set_xlabel("α (deadline level)")
ax.set_ylabel("ILP Punctuality Probability")
ax.set_ylim(0, 1.05)
ax.set_title("ILP Punctuality vs Deadline Level")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Save Results

In [ ]:
OUTPUT_DIR = Path("notebook/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

result.save_csv(OUTPUT_DIR / "results.csv")
print(f"Results saved to: {OUTPUT_DIR / 'results.csv'}")

## 7. Advanced — From In-Memory Data

No pre-generated directory needed. Build a dataset from a NetworkX graph and numpy arrays.

In [ ]:
import networkx as nx

# Build a 4-node diamond network
G = nx.DiGraph()
G.add_edge(0, 1)
G.add_edge(0, 2)
G.add_edge(1, 3)
G.add_edge(2, 3)

# Generate synthetic lognormal travel times: 500 samples × 4 edges
rng = np.random.default_rng(42)
W_toy = rng.lognormal(mean=3.0, sigma=0.8, size=(500, 4))

# Build dataset and config
toy_ds = RSPDataset.from_arrays(G, W_toy, [(0, 3)], meta={"description": "toy diamond"})
toy_cfg = RSPConfig(methods=("ILP",), alphas=(0.5, 0.7), num_repeats=1, num_od_pairs=1)
toy_runner = RSPRunner(toy_ds, toy_cfg)

# Run and show path
toy_result = toy_runner.run(progress=False)
display(toy_result.by_alpha().round(4))

# Show path for α=0.5
case_toy = toy_runner.solve_case(repeat=0, od_idx=0, alpha=0.5)
if case_toy.ilp["path_x"] is not None:
    path_edges = [
        toy_ds.edge_order[j] for j in np.where(case_toy.ilp["path_x"] > 0.5)[0]
    ]
    print(
        f"\nILP path: {' → '.join(str(e[0]) for e in path_edges)} → {path_edges[-1][1]}"
    )
    print(f"Edges: {path_edges}")
    print(f"Punctuality: {case_toy.ilp['punctuality_prob']:.4f}")

---

**Notebook done.** Change `DATASET_PATH` and `config` in §2 to switch datasets or parameters.